In [ ]:
# Librerías que importamos
# pip install openmeteo-requests
# pip install requests-cache retry-requests numpy pandas

In [2]:
#Importamos las librerias
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

c:\Users\wwwpi\AppData\Local\Programs\Python\Python39\lib\site-packages\requests\__init__.py:102: RequestsDependencyWarning: urllib3 (2.20.907) or chardet (None)/charset_normalizer (2.0.7) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({})/charset_normalizer ({}) doesn't match a supported "


In [3]:
# Configurar la API con la caché y reintentos (necesario para el funcionamiento de la api)
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [4]:
# Hay que asegurarse que todas las variables de clima se encunetran enlistadas
# dentro de los parámteros 
# El orden de las variables (por hora o por día) están en orden
# el orden es importante y es como se muestra a continuación
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 20.6774,
	"longitude": -103.3475,
	"daily": ["temperature_2m_max", "temperature_2m_min", "daylight_duration", "sunshine_duration", "uv_index_max", "uv_index_clear_sky_max"],
	"hourly": ["temperature_2m", "relative_humidity_2m", "precipitation_probability", "precipitation", "wind_speed_10m", "wind_speed_80m", "wind_direction_80m", "wind_direction_120m", "wind_direction_10m", "wind_speed_120m", "wind_speed_180m", "wind_direction_180m", "dew_point_2m", "shortwave_radiation"],
	"timezone": "America/Denver",
	"past_days": 14,
	"forecast_days": 16,
}

In [5]:
# Respuesta o solicitud (request de la API)
responses = openmeteo.weather_api(url, params = params)

In [6]:
# Vemos la primera respuesta (hay que procesar varias locaciones con ayuda de un for o bucle)
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

Coordinates: 20.702985763549805°N -103.36361694335938°E
Elevation: 1554.0 m asl
Timezone: b'America/Denver'b'GMT-6'
Timezone difference to GMT+0: -21600s


In [7]:
# Procesamos los datos que van por horario (nótese que el orden de las variables sigue el mismo que el solicitado).
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(2).ValuesAsNumpy()
hourly_precipitation = hourly.Variables(3).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(4).ValuesAsNumpy()
hourly_wind_speed_80m = hourly.Variables(5).ValuesAsNumpy()
hourly_wind_direction_80m = hourly.Variables(6).ValuesAsNumpy()
hourly_wind_direction_120m = hourly.Variables(7).ValuesAsNumpy()
hourly_wind_direction_10m = hourly.Variables(8).ValuesAsNumpy()
hourly_wind_speed_120m = hourly.Variables(9).ValuesAsNumpy()
hourly_wind_speed_180m = hourly.Variables(10).ValuesAsNumpy()
hourly_wind_direction_180m = hourly.Variables(11).ValuesAsNumpy()
hourly_dew_point_2m = hourly.Variables(12).ValuesAsNumpy()
hourly_shortwave_radiation = hourly.Variables(13).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

In [8]:
# Nombramos las variables
hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["precipitation"] = hourly_precipitation
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_speed_80m"] = hourly_wind_speed_80m
hourly_data["wind_direction_80m"] = hourly_wind_direction_80m
hourly_data["wind_direction_120m"] = hourly_wind_direction_120m
hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
hourly_data["wind_speed_120m"] = hourly_wind_speed_120m
hourly_data["wind_speed_180m"] = hourly_wind_speed_180m
hourly_data["wind_direction_180m"] = hourly_wind_direction_180m
hourly_data["dew_point_2m"] = hourly_dew_point_2m
hourly_data["shortwave_radiation"] = hourly_shortwave_radiation

In [9]:
#Imprimimos el data frame
hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)


Hourly data
                          date  temperature_2m  relative_humidity_2m  \
0   2026-05-12 00:00:00-06:00       20.750000             60.969944   
1   2026-05-12 01:00:00-06:00       19.200001             72.326988   
2   2026-05-12 02:00:00-06:00       18.500000             78.294022   
3   2026-05-12 03:00:00-06:00       18.200001             79.780525   
4   2026-05-12 04:00:00-06:00       18.049999             80.017624   
..                        ...             ...                   ...   
715 2026-06-10 19:00:00-06:00       31.955999             15.000000   
716 2026-06-10 20:00:00-06:00       30.806000             15.000000   
717 2026-06-10 21:00:00-06:00       29.406000             19.000000   
718 2026-06-10 22:00:00-06:00       27.656000             29.000000   
719 2026-06-10 23:00:00-06:00       25.656000             42.000000   

     precipitation_probability  precipitation  wind_speed_10m  wind_speed_80m  \
0                         53.0            0.0       

In [10]:
# Obtenemos el CSV
hourly_dataframe.to_csv("variables_por_hora")

In [16]:
# Procesamos los datos diarios (el orden de las variables son como solicitamos)
daily = response.Daily()
daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
daily_daylight_duration = daily.Variables(2).ValuesAsNumpy()
daily_sunshine_duration = daily.Variables(3).ValuesAsNumpy()
daily_uv_index_max = daily.Variables(4).ValuesAsNumpy()
daily_uv_index_clear_sky_max = daily.Variables(5).ValuesAsNumpy()

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	).tz_convert(response.Timezone().decode())
}

daily_data["temperature_2m_max"] = daily_temperature_2m_max
daily_data["temperature_2m_min"] = daily_temperature_2m_min
daily_data["daylight_duration"] = daily_daylight_duration
daily_data["sunshine_duration"] = daily_sunshine_duration
daily_data["uv_index_max"] = daily_uv_index_max
daily_data["uv_index_clear_sky_max"] = daily_uv_index_clear_sky_max

daily_dataframe = pd.DataFrame(data = daily_data)
print("\nDaily data\n", daily_dataframe)


Daily data
                         date  temperature_2m_max  temperature_2m_min  \
0  2026-05-12 00:00:00-06:00           29.600000           16.900000   
1  2026-05-13 00:00:00-06:00           30.150000           18.799999   
2  2026-05-14 00:00:00-06:00           32.299999           19.000000   
3  2026-05-15 00:00:00-06:00           33.849998           19.650000   
4  2026-05-16 00:00:00-06:00           33.150002           18.299999   
5  2026-05-17 00:00:00-06:00           33.200001           18.850000   
6  2026-05-18 00:00:00-06:00           32.450001           19.450001   
7  2026-05-19 00:00:00-06:00           32.299999           19.400000   
8  2026-05-20 00:00:00-06:00           33.099998           20.250000   
9  2026-05-21 00:00:00-06:00           32.450001           19.100000   
10 2026-05-22 00:00:00-06:00           32.250000           20.100000   
11 2026-05-23 00:00:00-06:00           31.100000           19.299999   
12 2026-05-24 00:00:00-06:00           33.650002   

In [17]:
#Limipiamos el formato de las fechas
daily_dataframe["date"] = daily_dataframe["date"].dt.strftime("%Y-%m-%d")

In [18]:
# Extraemos el csv
daily_dataframe.to_csv("datos_diarios")

In [ ]:
# Encontramos los valores nulos por campo (si es que hay)
datos_diarios_df = pd.read_csv("datos_diarios")

null_counts = datos_diarios_df.isnull().sum()
null_fields = null_counts[null_counts > 0]

print("Null values by field:")
print("=" * 50)
if len(null_fields) > 0:
    for field, count in null_fields.items():
        print(f"{field}: {count} nulls")
else:
    print("No null values found in any field")


Null values by field:
No null values found in any field


In [ ]:
# Encontramos los valores nulos por campo (si es que hay)
datos_diarios_df = pd.read_csv("variables_por_hora")

null_counts = datos_diarios_df.isnull().sum()

null_fields = null_counts[null_counts > 0]

print("Null values by field:")
print("=" * 50)
if len(null_fields) > 0:
    for field, count in null_fields.items():
        print(f"{field}: {count} nulls")
else:
    print("No null values found in any field")

Null values by field:
wind_speed_180m: 17 nulls
wind_direction_180m: 17 nulls


In [ ]:
# encontrar duplicados en el archivo de datos diarios
datos_diarios_df = pd.read_csv("datos_diarios")


duplicated_rows = datos_diarios_df[datos_diarios_df.duplicated(keep=False)].sort_values(by=list(datos_diarios_df.columns))

print(f"Total rows in file: {len(datos_diarios_df)}")
print(f"Duplicate rows found: {len(duplicated_rows)}")
print("=" * 50)

if len(duplicated_rows) > 0:
    print("\nDuplicate rows:")
    print(duplicated_rows)
else:
    print("\nNo duplicate rows found!")


Total rows in file: 30
Duplicate rows found: 0

No duplicate rows found!


In [22]:
# encontrar duplicados en el archivo de datos diarios
datos_diarios_df = pd.read_csv("variables_por_hora")


duplicated_rows = datos_diarios_df[datos_diarios_df.duplicated(keep=False)].sort_values(by=list(datos_diarios_df.columns))

print(f"Total rows in file: {len(datos_diarios_df)}")
print(f"Duplicate rows found: {len(duplicated_rows)}")
print("=" * 50)

if len(duplicated_rows) > 0:
    print("\nDuplicate rows:")
    print(duplicated_rows)
else:
    print("\nNo duplicate rows found!")

Total rows in file: 720
Duplicate rows found: 0

No duplicate rows found!


In [ ]:
# todo junto en una sola función para analizar la calidad de los datos (valores nulos y filas duplicadas)
def analyze_csv_quality(filename):
    """
    Analyze a CSV file to report null values and duplicate rows.
    
    Parameters:
    filename (str): Path or filename of the CSV to analyze
    """
    df = pd.read_csv(filename)
    
    print(f"\n{'='*60}")
    print(f"DATA QUALITY REPORT: {filename}")
    print(f"{'='*60}")
    
    # Total rows and columns
    print(f"\nFile Statistics:")
    print(f"  Total Rows: {len(df)}")
    print(f"  Total Columns: {len(df.columns)}")
    
    # Null values analysis
    print(f"\n{'-'*60}")
    print("NULL VALUES ANALYSIS:")
    print(f"{'-'*60}")
    
    null_counts = df.isnull().sum()
    null_fields = null_counts[null_counts > 0]
    
    if len(null_fields) > 0:
        total_nulls = null_fields.sum()
        print(f"Total null values found: {total_nulls}\n")
        for field, count in null_fields.items():
            percentage = (count / len(df)) * 100
            print(f"  • {field}: {count} nulls ({percentage:.2f}%)")
    else:
        print("✓ No null values found in any field")
    
    # Duplicate rows analysis
    print(f"\n{'-'*60}")
    print("DUPLICATE ROWS ANALYSIS:")
    print(f"{'-'*60}")
    
    duplicate_count = df.duplicated().sum()
    total_duplicates = len(df[df.duplicated(keep=False)])
    
    if duplicate_count > 0:
        print(f"Duplicate row instances found: {duplicate_count}")
        print(f"Total rows involved in duplicates: {total_duplicates}")
    else:
        print("✓ No duplicate rows found")
    
    print(f"\n{'='*60}\n")




DATA QUALITY REPORT: datos_diarios

File Statistics:
  Total Rows: 30
  Total Columns: 8

------------------------------------------------------------
NULL VALUES ANALYSIS:
------------------------------------------------------------
✓ No null values found in any field

------------------------------------------------------------
DUPLICATE ROWS ANALYSIS:
------------------------------------------------------------
✓ No duplicate rows found



DATA QUALITY REPORT: variables_por_hora

File Statistics:
  Total Rows: 720
  Total Columns: 16

------------------------------------------------------------
NULL VALUES ANALYSIS:
------------------------------------------------------------
Total null values found: 34

  • wind_speed_180m: 17 nulls (2.36%)
  • wind_direction_180m: 17 nulls (2.36%)

------------------------------------------------------------
DUPLICATE ROWS ANALYSIS:
------------------------------------------------------------
✓ No duplicate rows found




In [24]:
#analizamos para la data diaria

analyze_csv_quality("datos_diarios")


DATA QUALITY REPORT: datos_diarios

File Statistics:
  Total Rows: 30
  Total Columns: 8

------------------------------------------------------------
NULL VALUES ANALYSIS:
------------------------------------------------------------
✓ No null values found in any field

------------------------------------------------------------
DUPLICATE ROWS ANALYSIS:
------------------------------------------------------------
✓ No duplicate rows found




In [25]:
# Analizamos para la variable por hora

analyze_csv_quality("variables_por_hora")



DATA QUALITY REPORT: variables_por_hora

File Statistics:
  Total Rows: 720
  Total Columns: 16

------------------------------------------------------------
NULL VALUES ANALYSIS:
------------------------------------------------------------
Total null values found: 34

  • wind_speed_180m: 17 nulls (2.36%)
  • wind_direction_180m: 17 nulls (2.36%)

------------------------------------------------------------
DUPLICATE ROWS ANALYSIS:
------------------------------------------------------------
✓ No duplicate rows found


